# 🎬 Otaku AI Viral Shorts Studio (Google Colab Edition)
### Fully Autonomous Cloud Pipeline Powered by **Gemini 2.5 Flash Multimodal Vision**

**100% Free Cloud Production:** Run directly from your phone browser or laptop without keeping your computer on.
- 🧠 **Multimodal LLM Director:** Samples candidate frames, detects beat drops & visual hooks, and chooses precision cut timestamps.
- 💎 **DSP Renderer:** 1080×1920 edge-to-edge framing, dynamic dual-layer ambient blur, Krylo countdown ladder, 48 kHz audio mastering.
- 🚀 **YouTube Studio Scheduler:** Automated canary upload, Content-ID verification, and primetime scheduling.

## ⚙️ 1. Studio Configuration & Controls
Set your desired topic or leave blank for automated virality scouting.

In [ ]:
#@title 🎛️ Production Settings { run: "auto" }
TOPIC = "The Mentalist" #@param {type:"string"}
CLIP_DURATION_SEC = 16 #@param {type:"slider", min:10, max:25, step:1}
USE_LLM_VISION_CUTS = True #@param {type:"boolean"}
UPLOAD_TO_YOUTUBE = False #@param {type:"boolean"}
GEMINI_API_KEY = "" #@param {type:"string"}

import os
if not GEMINI_API_KEY:
    try:
        from google.colab import userdata
        GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    except Exception:
        pass

if GEMINI_API_KEY:
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
    print(f"✅ Gemini 2.5 Flash Vision Enabled (Key loaded)")
else:
    print("⚠️ No Gemini API key provided; falling back to heuristic audio/visual cuts.")


## 📦 2. Install Cloud Dependencies & Setup Engine

In [ ]:
#@title 🚀 Install Dependencies (Takes ~20s)
!apt-get update -qq && apt-get install -y -qq ffmpeg fonts-dejavu-core > /dev/null
!pip install -q yt-dlp pillow opencv-python-headless google-generativeai google-api-python-client google-auth-oauthlib

import subprocess, sys
# Clone or pull latest Otaku repo if running in a fresh Colab VM
if not os.path.exists('dsp_renderer.py'):
    print("Cloning latest engine from GitHub...")
    !git clone -q https://github.com/sp25126/MOVIE-SERIES---AUTO.git otaku_repo
    %cd otaku_repo
else:
    !git pull -q

print("✅ Cloud Environment Ready!")


## 🧠 3. Scout Viral Clips & Multimodal Frame Analysis
Gemini 2.5 Flash analyzes sequential video frames to detect the peak energy moments, optimal cut points, and rank candidates.

In [ ]:
#@title 🔍 Scout & Analyze Candidate Clips
import pathlib, json
from clip_harvester import scout_youtube, download_clip
from llm_director import curate_and_rank_clips, analyze_clip_frames

topic_query = TOPIC.strip() or "The Mentalist"
print(f"🔎 Scouting top trending fan edits for: '{topic_query}'...")

raw_candidates = scout_youtube(f"{topic_query} edit shorts", count=5, ignore_history=True)
if not raw_candidates:
    raw_candidates = scout_youtube(f"{topic_query} bad ass edit", count=5, ignore_history=True)

print(f"Found {len(raw_candidates)} candidate edits. Downloading high-res source streams...")
input_dir = pathlib.Path('input')
input_dir.mkdir(exist_ok=True)

downloaded = []
for cand in raw_candidates[:4]:
    p = download_clip(cand, out_dir=input_dir)
    if p and p.exists():
        cand['local_path'] = str(p)
        downloaded.append(cand)
        print(f"  ✓ Downloaded: {cand['title'][:40]} ({p.stat().st_size / 1e6:.1f}MB)")

if not downloaded:
    raise RuntimeError("Failed to harvest candidate clips. Check network or search query.")

print("\n🧠 Launching Gemini 2.5 Flash Multimodal Vision Inspection...")
curated_clips = curate_and_rank_clips(downloaded, topic=topic_query, target_count=3, api_key=os.environ.get('GEMINI_API_KEY'))

print("\n--- AI EDITORIAL RANKING REPORT ---")
for i, c in enumerate(curated_clips, 1):
    ai = c.get('ai_analysis', {})
    cut = c.get('optimal_cut', {})
    print(f"🥇 Rank #{i}: {c.get('title')[:35]}")
    print(f"   • Quality Score: {c.get('quality_score')}/10 | Label: {c.get('short_label')}")
    print(f"   • Optimal AI Cut: {cut.get('start_sec', 0.0)}s -> {cut.get('end_sec', 0.0)}s ({cut.get('duration')}s)")
    print(f"   • Hook: {ai.get('hook_description', 'N/A')[:60]}...")
    print(f"   • Climax: {ai.get('climax_description', 'N/A')[:60]}...\n")


## 🎨 4. Render Master Short (1080×1920 Edge-to-Edge)
Executes precision trimming on LLM cut points, adds dynamic ambient blurred background, overlays the countdown ladder, and masters 48 kHz broadcast audio.

In [ ]:
#@title 🎬 Render Video & Preview
import pathlib
from dsp_renderer import render_edit_ranking
import qa_gate
from IPython.display import Video, display

out_mp4 = pathlib.Path(f"output/{topic_query.lower().replace(' ', '_')}_ai_master.mp4")
out_mp4.parent.mkdir(exist_ok=True)

print(f"Rendering master short to {out_mp4}...")
render_edit_ranking(
    topic=topic_query,
    clips=curated_clips,
    output_mp4=out_mp4,
    clip_dur=float(CLIP_DURATION_SEC),
)

ok = qa_gate.verify_video(out_mp4)
print(f"QA Gate Result: {ok}")
print(f"File Size: {out_mp4.stat().st_size / 1e6:.2f} MB")

# Display video preview directly in Colab!
display(Video(str(out_mp4), embed=True, width=360))


## 🚀 5. YouTube Studio Canary Upload & Scheduling (Optional)
Uploads as a private canary, checks for copyright claims, and schedules for the next 7:30 PM IST primetime slot.

In [ ]:
#@title 📤 Upload & Schedule to YouTube Studio
if not UPLOAD_TO_YOUTUBE:
    print("ℹ️ UPLOAD_TO_YOUTUBE is disabled. Master video is saved in output/ ready for download.")
else:
    import metadata_uploader
    meta = metadata_uploader.build_metadata(topic_query, 'edit_ranking', curated_clips)
    print("Generated AI Metadata:")
    print(f"  Title: {meta['title']}")
    print(f"  Slot:  {meta['recommended_slot_utc']}")
    
    print("\nUploading private canary...")
    res = metadata_uploader.upload_private_canary_and_schedule(
        video_path=out_mp4,
        meta=meta,
        canary_wait_sec=90
    )
    print(f"✅ Successfully Scheduled! https://youtube.com/shorts/{res.get('video_id')}")
